# Scalable Website Screenshot Parser Development

Use this notebook to step through the most recent Scalable broker parser against a new website screenshot pattern.
Each parsing stage is broken into individual cells. Failures show a full traceback without aborting the notebook.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import traceback
from pathlib import Path

from fintl.accounts_etl.scalable import broker20260309 as parser
from fintl.accounts_etl.schemas import (
    Case,
    ProviderEnum,
    ScalableBrokerParserEnum,
    ServiceEnum,
)

In [ ]:
BASE_DIR = Path(".../broker-artefacts")
FILE_PATH = BASE_DIR / "Screenshot 2026-03-02 at 14.30.53.png"  # <-- edit this

FILE_PATH.exists()

In [ ]:
CASE = Case(
    provider=ProviderEnum.scalable.value,
    service=ServiceEnum.broker.value,
    parser=ScalableBrokerParserEnum.broker20260309.value,
)

print(f"Target file : {FILE_PATH}")
print(f"File exists : {FILE_PATH.exists()}")
print(f"Case        : {CASE}")

## File info

In [ ]:
try:
    size_kb = FILE_PATH.stat().st_size / 1024
    print(f"Size     : {size_kb:.1f} KB")

except Exception:
    traceback.print_exc()

## `check_if_parser_applies`

In [ ]:
try:
    result = parser.check_if_parser_applies(FILE_PATH)
    print(f"check_if_parser_applies → {result}")
except Exception:
    traceback.print_exc()

## `extract_balance` step-by-step

Each sub-cell mirrors one step inside `broker20260309.extract_balance`.

In [ ]:
# Parse date
try:
    date = parser.get_date_from_string(FILE_PATH.name)
    print(f"Date    : {date}")
except Exception:
    traceback.print_exc()

In [ ]:
# Parse image with instructor & ollama
try:
    extraction_client = parser._get_ollama_client(model="qwen3.5:27b")
    assert extraction_client is not None, "ollama client setup failed"

    extraction = parser._get_lm_extraction(FILE_PATH, extraction_client)

    print(f"Balance : {extraction.amount} {extraction.currency}")

    print("\nsoup parsed OK")

except Exception:
    traceback.print_exc()

In [ ]:
# Construct BalanceInfo
try:
    from fintl.accounts_etl.schemas import BalanceInfo

    balance = BalanceInfo(
        date=date,
        amount=extraction.amount,
        currency=extraction.currency,
        provider=CASE.provider,
        service=CASE.service,
        parser=CASE.parser,
        file=str(FILE_PATH),
    )
    print(balance)
except Exception:
    traceback.print_exc()

## `extract_transactions` step-by-step

`broker20231028` delegates to `broker0.extract_transactions`, which currently returns an **empty** DataFrame (no image parsing required yet).
If Scalable adds transaction data to the image in a future format change, this section will need expanding.

In [ ]:
try:
    transactions = parser.extract_transactions()
    print(f"Shape  : {transactions.shape}")
    print(f"Schema : {transactions.schema}")
    print()
    print(transactions)
except Exception:
    traceback.print_exc()

## End-to-end smoke test

In [ ]:
from fintl.accounts_etl.schemas import OllamaConfig

try:
    ollama_cfg = OllamaConfig(model="qwen3.5:27b")
    txns, bal = parser.parse_image_file(CASE, FILE_PATH, ollama_config=ollama_cfg)
    print("parse_image_file succeeded")
    print()
    print(f"transactions shape  : {txns.shape}")
    print(f"transactions schema : {txns.schema}")
    print()
    print(f"balance : {bal}")
except Exception:
    traceback.print_exc()